In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.tsa.seasonal import STL

from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().resolve().parent))
print(sys.executable)


/Users/yuanxinyi/Desktop/panel-forcast-kit/.venv/bin/python


# Raw Data Acquisition & Preview

In [5]:
project_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "train.csv").exists()),
    Path.cwd(),
)
csv_path = project_root / "data" / "raw" / "train.csv"

df = pd.read_csv(
    csv_path,
    parse_dates=["date"],
) 

print("File loaded from:", csv_path)
print("Shape:", df.shape)
print("\nPreview:")
print(df.head())
print("\nDate range:", df["date"].min(), "->", df["date"].max())
print(
    "\nNumber of stores:", df["store"].nunique(),
    "\nNumber of items:", df["item"].nunique(),
) 

File loaded from: /Users/yuanxinyi/Desktop/panel-forcast-kit/data/raw/train.csv
Shape: (913000, 4)

Preview:
        date  store  item  sales
0 2013-01-01      1     1     13
1 2013-01-02      1     1     11
2 2013-01-03      1     1     14
3 2013-01-04      1     1     13
4 2013-01-05      1     1     10

Date range: 2013-01-01 00:00:00 -> 2017-12-31 00:00:00

Number of stores: 10 
Number of items: 50


# Data Quality Check

In [8]:
print("=== Basics ===")

print(f"\nMissing value counts by column: \n{df.isnull().sum()}")
print(f"\nNumber of records with negative sales: {(df['sales'] < 0).sum()}")
print(f"\nNumber of records with zero sales: {(df['sales'] == 0).sum()}")
print(
    f"\nNumber of duplicate records (same date, store, and item): "
    f"{df.duplicated(subset=['date', 'store', 'item']).sum()}"
)

=== Basics ===

Missing value counts by column: 
date     0
store    0
item     0
sales    0
dtype: int64

Number of records with negative sales: 0

Number of records with zero sales: 1

Number of duplicate records (same date, store, and item): 0


In [9]:
print("=== Descriptive Statistics ===")
print(f"\nDescriptive Statistics for Sales:\n{df['sales'].describe()}")

daily = df.groupby("date")["sales"].sum().sort_index()
stl = STL(daily, period=365, robust=True).fit()
resid = stl.resid 
print(f"\nDescriptive Statistics for Residuals:\n{resid.describe()}")

threshold = 3 * resid.std()
anomalies = resid[resid.abs() > threshold]
print(f"\n=== Residual-Based Anomalies (deviations > 3σ after removing trend and seasonality) ===")
print(f"Number of anomalous days: {len(anomalies)} / {len(daily)} days")
print(f"\nAnomalous dates and their residuals, sorted by magnitude of deviation:")
for date, r in anomalies.reindex(anomalies.abs().sort_values(ascending=False).index).items():
    print(f"  {date.date()}  residual {r:+8.0f}  (total sales on that day: {daily[date]:.0f})")

=== Descriptive Statistics ===

Descriptive Statistics for Sales:
count    913000.000000
mean         52.250287
std          28.801144
min           0.000000
25%          30.000000
50%          47.000000
75%          70.000000
max         231.000000
Name: sales, dtype: float64

Descriptive Statistics for Residuals:
count     1826.000000
mean       -42.748319
std       3417.976546
min     -18574.069155
25%       -204.954097
50%          8.948734
75%        250.409887
max      11003.439251
Name: resid, dtype: float64

=== Residual-Based Anomalies (deviations > 3σ after removing trend and seasonality) ===
Number of anomalous days: 47 / 1826 days

Anomalous dates and their residuals, sorted by magnitude of deviation:
  2017-07-10  residual   -18574  (total sales on that day: 29723)
  2017-07-17  residual   -18388  (total sales on that day: 29959)
  2017-07-03  residual   -18228  (total sales on that day: 29775)
  2017-07-24  residual   -17756  (total sales on that day: 30041)
  2017-06-05 

In [13]:
print("=== Extreme Anomalies ===")

if "dayofweek" not in df.columns:
    df["dayofweek"] = df["date"].dt.dayofweek 
    print("The 'dayofweek' column was added.")
else:
    print("The 'dayofweek' column already exists.")

df["is_weekend"] = df["dayofweek"].isin([5, 6])
grp = df.groupby(["store", "item", "is_weekend"])["sales"]  

df["grp_mean"] = grp.transform("mean")
df["grp_std"] = grp.transform("std")
df["z"] = (df["sales"] - df["grp_mean"]) / df["grp_std"]

outliers = df[df["z"].abs() > 5] 
print(f"Number of extreme anomalies (|z| > 5): {len(outliers)} / {len(df)} ({len(outliers)/len(df)*100:.4f}%)")
print(f"\nOverall sales range: {df['sales'].min()} ~ {df['sales'].max()}")
print(f"Negative values: {(df['sales'] < 0).sum()},  Zero values: {(df['sales'] == 0).sum()}")

if len(outliers) > 0:
    print("\nTop extreme cases:")
    print(outliers[["date", "store", "item", "sales", "grp_mean", "z"]].head(10).to_string())
else:
    print("\n-> No 5-sigma extreme anomalies were detected; the data looks clean.")

=== Extreme Anomalies ===
The 'dayofweek' column already exists.
Number of extreme anomalies (|z| > 5): 2 / 913000 (0.0002%)

Overall sales range: 0 ~ 231
Negative values: 0,  Zero values: 1

Top extreme cases:
             date  store  item  sales   grp_mean         z
1639   2017-06-28      1     1     50  18.605061  5.128499
737526 2017-07-07      4    41     57  21.710890  5.114521


# Conlusion
Data is clean